In [ ]:
import torch

print("====================================")
print("SYSTEM CHECK")
print("====================================")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is NOT available.")

SYSTEM CHECK
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
import os
import json
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from datasets import load_from_disk

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, f1_score

print("All imports completed successfully.")

All imports completed successfully.


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", device)

Selected device: cuda


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/SLM_KD_Research"

os.makedirs(PROJECT_ROOT, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

Project root:
/content/drive/MyDrive/SLM_KD_Research


In [ ]:
folders = [
    "data",
    "data/processed",
    "models",
    "models/baseline_student",
    "models/kd",
    "models/ablation",
    "results",
    "results/ablation",
    "results/error_analysis",
    "results/final",
    "results/plots",
    "logs",
    "notebooks"
]

for folder in folders:
    os.makedirs(
        os.path.join(PROJECT_ROOT, folder),
        exist_ok=True
    )

print("All project folders created.")

All project folders created.


In [ ]:
print("Current Colab directory:")
print(os.getcwd())

print("\nFiles/folders in /content:")
print(os.listdir("/content"))

Current Colab directory:
/content

Files/folders in /content:
['.config', 'drive', 'sample_data']


In [ ]:
matches = []

for root, dirs, files in os.walk("/content"):
    if "raw_dataset" in dirs:
        matches.append(
            os.path.join(root, "raw_dataset")
        )

print("Found raw_dataset locations:")

for path in matches:
    print(path)

Found raw_dataset locations:


In [ ]:
import os

print("Folders inside /content/drive:")
print(os.listdir("/content/drive"))

Folders inside /content/drive:
['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']


In [ ]:
import os

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "raw_dataset" in dirs:
        matches.append(os.path.join(root, "raw_dataset"))

print("Found raw_dataset locations:")

for path in matches:
    print(path)

Found raw_dataset locations:


In [ ]:
import os

print("========== MY DRIVE ==========")

for item in os.listdir("/content/drive/MyDrive"):
    print(item)

========== MY DRIVE ==========
Colab Notebooks
Untitled document (6).gdoc
IMG-20250821-WA0020.jpg
Screenshot_2026-01-25-13-00-45-06_6012fa4d4ddec268fc5c7112cbb265e7.jpg
Untitled document (5).gdoc
Untitled document (4).gdoc
Untitled document (3).gdoc
ASIYA SAFWANA S_GOOGLE _STUDENT _REASERCH _2026.gdoc
ASIYA_SAFWANA_S GOOGLE _STUDENT _2026.gdoc
Completion Certificate _ SkillsBuild.pdf
IMG_20260312_205200.jpg
Screenshot_2026-03-13-21-07-24-66_f9ee0578fe1cc94de7482bd41accb329.jpg
Screenshot_2026-03-14-21-13-01-79_f9ee0578fe1cc94de7482bd41accb329.jpg
Screenshot_2026-03-15-21-36-18-47_f9ee0578fe1cc94de7482bd41accb329.jpg
ASIYA SAFWANA S
Internship contract - Communique x Student Ambassador.docx
IMG-20260521-WA0018.jpg
Class 8 AdisthanaPaadaavali .docx
.mp4
Screenshot_2026-06-14-22-58-57-58_1c337646f29875672b5a61192b9010f9.jpg
Screenshot_2026-06-14-23-07-21-80_99c04817c0de5652397fc8b56c3b3817.jpg
Screenshot_2026-06-14-23-07-26-92_99c04817c0de5652397fc8b56c3b3817.jpg
Screenshot_2026-06-14-23-

In [ ]:
from datasets import load_dataset

print("Downloading SST-2 dataset...")

dataset = load_dataset(
    "nyu-mll/glue",
    "sst2"
)

print("\nDataset downloaded successfully!")
print(dataset)

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]


Dataset downloaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [ ]:
print("====================================")
print("DATASET INSPECTION")
print("====================================")

print(dataset)

print("\nDataset splits:")
print(dataset.keys())

for split in dataset.keys():
    print(f"\n{split}:")
    print("Columns:", dataset[split].column_names)
    print("Number of rows:", len(dataset[split]))

print("\nSample training example:")
print(dataset["train"][0])

DATASET INSPECTION
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

Dataset splits:
dict_keys(['train', 'validation', 'test'])

train:
Columns: ['sentence', 'label', 'idx']
Number of rows: 67349

validation:
Columns: ['sentence', 'label', 'idx']
Number of rows: 872

test:
Columns: ['sentence', 'label', 'idx']
Number of rows: 1821

Sample training example:
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [ ]:
from transformers import AutoTokenizer

STUDENT_MODEL = "google/bert_uncased_L-2_H-128_A-2"

print("Student model:", STUDENT_MODEL)
print("Loading tokenizer...")

student_tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL
)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", student_tokenizer.vocab_size)

Student model: google/bert_uncased_L-2_H-128_A-2
Loading tokenizer...


config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 30522


In [ ]:
# ==========================================
# TOKENIZER TEST
# ==========================================

sample_text = dataset["train"][0]["sentence"]

print("Original sentence:")
print(sample_text)

print("\n------------------------------------")

tokens = student_tokenizer.tokenize(sample_text)

print("Tokens:")
print(tokens)

print("\n------------------------------------")

token_ids = student_tokenizer(
    sample_text,
    truncation=True,
    max_length=128
)

print("Token IDs:")
print(token_ids["input_ids"])

print("\nNumber of tokens:")
print(len(token_ids["input_ids"]))

Original sentence:
hide new secretions from the parental units 

------------------------------------
Tokens:
['hide', 'new', 'secret', '##ions', 'from', 'the', 'parental', 'units']

------------------------------------
Token IDs:
[101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102]

Number of tokens:
10


In [ ]:
# ==========================================
# FULL SST-2 TOKENIZATION
# ==========================================

MAX_LENGTH = 128

def tokenize_dataset(example):
    return student_tokenizer(
        example["sentence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

print("Tokenizing SST-2 dataset...")
print("Maximum sequence length:", MAX_LENGTH)

tokenized_dataset = dataset.map(
    tokenize_dataset,
    batched=True
)

print("\nTokenization completed successfully!")

print("\nTokenized dataset:")
print(tokenized_dataset)

print("\nColumns in training set:")
print(tokenized_dataset["train"].column_names)

Tokenizing SST-2 dataset...
Maximum sequence length: 128


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]


Tokenization completed successfully!

Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})

Columns in training set:
['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# ==========================================
# PREPARE DATASET FOR MODEL TRAINING
# ==========================================

model_columns = [
    "input_ids",
    "token_type_ids",
    "attention_mask",
    "label"
]

tokenized_dataset.set_format(
    type="torch",
    columns=model_columns
)

print("Dataset format changed to PyTorch.")

print("\nFirst training example:")
print(tokenized_dataset["train"][0])

Dataset format changed to PyTorch.

First training example:
{'label': tensor(0), 'input_ids': tensor([  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}


In [ ]:
# ==========================================
# DATA COLLATOR + DATALOADERS
# ==========================================

from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

data_collator = DataCollatorWithPadding(
    tokenizer=student_tokenizer
)

BATCH_SIZE = 16

train_loader = DataLoader(
    tokenized_dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

validation_loader = DataLoader(
    tokenized_dataset["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    tokenized_dataset["test"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print("DataLoaders created successfully!")

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

DataLoaders created successfully!
Training batches: 4210
Validation batches: 55
Test batches: 114


In [ ]:
# ==========================================
# INSPECT ONE TRAINING BATCH
# ==========================================

batch = next(iter(train_loader))

print("Batch keys:")
print(batch.keys())

print("\nInput IDs shape:")
print(batch["input_ids"].shape)

print("\nAttention mask shape:")
print(batch["attention_mask"].shape)

print("\nLabels shape:")
print(batch["labels"].shape)

print("\nLabels:")
print(batch["labels"])

print("\nFirst example input IDs:")
print(batch["input_ids"][0])

print("\nFirst example attention mask:")
print(batch["attention_mask"][0])

Batch keys:
KeysView({'input_ids': tensor([[  101,  2062,  8669,  1010,  2062,  6832,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [  101,  2009,  1005,  1055, 10889, 11519,  1010,  3391,  2005,  1037,
          7891, 18932,  1999,  1037,  2186,  1012,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [  101,  2003,  1037,  3407,  1010,  2132,  2100, 18414, 19661,  1997,
          2245,  1998, 20957,  1010,  2019,  9577,  5021, 18457,  2008,  4821,
          2522,  5886,  2229,  2046,  1037, 22856,  1998,  3052, 17904,  2135,
          5541,  2143,   102],
        [  101,  2009,  2453,  2074,  2022,  2488, 10897,  2000,  1037,  2305,
          1999,  1996,  2542,  2282,  2084,  1037,  2305,  2012,  1996,  5691,
   

In [ ]:
# ==========================================
# LOAD BERT-TINY STUDENT MODEL
# ==========================================

from transformers import AutoModelForSequenceClassification

NUM_LABELS = 2

print("Loading Student model...")
print("Model:", STUDENT_MODEL)

student_model = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL,
    num_labels=NUM_LABELS
)

print("\nStudent model loaded successfully!")

print("Number of labels:", student_model.config.num_labels)
print("Hidden size:", student_model.config.hidden_size)
print("Number of layers:", student_model.config.num_hidden_layers)

Loading Student model...
Model: google/bert_uncased_L-2_H-128_A-2


model.safetensors: reconstructing file:   0%|          |  0.00B / 17.7MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Student model loaded successfully!
Number of labels: 2
Hidden size: 128
Number of layers: 2


In [ ]:
# ==========================================
# STUDENT MODEL PARAMETER COUNT
# ==========================================

total_params = sum(
    p.numel()
    for p in student_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in student_model.parameters()
    if p.requires_grad
)

print("====================================")
print("STUDENT MODEL PARAMETERS")
print("====================================")

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

print("\nTotal parameters in millions:",
      total_params / 1_000_000)

print("Trainable parameters in millions:",
      trainable_params / 1_000_000)

STUDENT MODEL PARAMETERS
Total parameters: 4386178
Trainable parameters: 4386178

Total parameters in millions: 4.386178
Trainable parameters in millions: 4.386178


In [ ]:
# ==========================================
# STUDENT MODEL SANITY CHECK
# ==========================================

print("Moving Student model to device...")

student_model = student_model.to(device)

print("Model device:", next(student_model.parameters()).device)

print("\nRunning one forward pass...")

student_model.eval()

with torch.no_grad():
    outputs = student_model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        labels=batch["labels"].to(device)
    )

print("\nForward pass successful!")

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)
print("Logits:")
print(outputs.logits)

Moving Student model to device...
Model device: cuda:0

Running one forward pass...

Forward pass successful!
Loss: 0.6882811784744263
Logits shape: torch.Size([16, 2])
Logits:
tensor([[ 0.0139,  0.2359],
        [-0.0156,  0.1563],
        [ 0.1067,  0.1743],
        [ 0.0967,  0.3385],
        [ 0.0311,  0.1793],
        [ 0.0626,  0.1442],
        [ 0.0457,  0.1992],
        [ 0.0808,  0.1948],
        [ 0.0682,  0.1708],
        [ 0.1486,  0.1591],
        [ 0.0647,  0.1507],
        [ 0.0941,  0.0948],
        [ 0.1060,  0.2792],
        [ 0.0511,  0.2280],
        [ 0.1922,  0.1534],
        [-0.0193,  0.1703]], device='cuda:0')


In [ ]:
# ==========================================
# BASELINE TRAINING CONFIGURATION
# ==========================================

LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

optimizer = torch.optim.AdamW(
    student_model.parameters(),
    lr=LEARNING_RATE
)

print("====================================")
print("BASELINE TRAINING CONFIGURATION")
print("====================================")

print("Student:", STUDENT_MODEL)
print("Dataset: SST-2")
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", NUM_EPOCHS)
print("Maximum sequence length:", MAX_LENGTH)
print("Optimizer: AdamW")
print("Device:", device)

print("\nOptimizer created successfully!")

BASELINE TRAINING CONFIGURATION
Student: google/bert_uncased_L-2_H-128_A-2
Dataset: SST-2
Batch size: 16
Learning rate: 2e-05
Epochs: 3
Maximum sequence length: 128
Optimizer: AdamW
Device: cuda

Optimizer created successfully!


In [ ]:
# ==========================================
# BASELINE STUDENT TRAINING
# ==========================================

from sklearn.metrics import accuracy_score, f1_score
import time

training_history = []

print("====================================")
print("STARTING BASELINE TRAINING")
print("====================================")

for epoch in range(NUM_EPOCHS):

    print(f"\n========== EPOCH {epoch + 1}/{NUM_EPOCHS} ==========")

    # ------------------------------
    # TRAINING
    # ------------------------------
    student_model.train()

    epoch_start_time = time.time()

    total_train_loss = 0.0

    for step, batch in enumerate(train_loader):

        # Move batch to GPU
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = student_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )

        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        total_train_loss += loss.item()

        # Progress message
        if (step + 1) % 500 == 0:
            print(
                f"Step {step + 1}/{len(train_loader)} "
                f"| Loss: {loss.item():.4f}"
            )

    average_train_loss = (
        total_train_loss / len(train_loader)
    )

    # ------------------------------
    # VALIDATION
    # ------------------------------
    student_model.eval()

    total_val_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in validation_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels = batch["labels"].to(device)

            outputs = student_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )

            total_val_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=-1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    average_val_loss = (
        total_val_loss / len(validation_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    val_f1 = f1_score(
        all_labels,
        all_predictions
    )

    epoch_time = time.time() - epoch_start_time

    # ------------------------------
    # SAVE EPOCH RESULTS
    # ------------------------------

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": average_train_loss,
        "validation_loss": average_val_loss,
        "validation_accuracy": val_accuracy,
        "validation_f1": val_f1,
        "epoch_time_seconds": epoch_time
    }

    training_history.append(epoch_result)

    print("\n------------------------------------")
    print(f"Epoch {epoch + 1} completed")
    print(f"Training Loss:      {average_train_loss:.4f}")
    print(f"Validation Loss:    {average_val_loss:.4f}")
    print(f"Validation Accuracy:{val_accuracy:.4f}")
    print(f"Validation F1:      {val_f1:.4f}")
    print(f"Epoch Time:         {epoch_time:.2f} seconds")
    print("------------------------------------")

print("\n====================================")
print("BASELINE TRAINING COMPLETED")
print("====================================")

STARTING BASELINE TRAINING

========== EPOCH 1/3 ==========
Step 500/4210 | Loss: 0.5413
Step 1000/4210 | Loss: 0.5121
Step 1500/4210 | Loss: 0.5647
Step 2000/4210 | Loss: 0.7343
Step 2500/4210 | Loss: 0.2894
Step 3000/4210 | Loss: 0.4109
Step 3500/4210 | Loss: 0.2659
Step 4000/4210 | Loss: 0.2356

------------------------------------
Epoch 1 completed
Training Loss:      0.4437
Validation Loss:    0.4421
Validation Accuracy:0.7993
Validation F1:      0.7958
Epoch Time:         90.78 seconds
------------------------------------

========== EPOCH 2/3 ==========
Step 500/4210 | Loss: 0.1540
Step 1000/4210 | Loss: 0.2973
Step 1500/4210 | Loss: 0.2181
Step 2000/4210 | Loss: 0.5137
Step 2500/4210 | Loss: 0.1813
Step 3000/4210 | Loss: 0.2462
Step 3500/4210 | Loss: 0.1606
Step 4000/4210 | Loss: 0.1735

------------------------------------
Epoch 2 completed
Training Loss:      0.3116
Validation Loss:    0.4350
Validation Accuracy:0.8119
Validation F1:      0.8194
Epoch Time:         67.78 seco

In [ ]:
# ==========================================
# SAVE BASELINE STUDENT CHECKPOINT
# ==========================================

import os
import json

BASELINE_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "baseline_student",
    "final"
)

os.makedirs(BASELINE_PATH, exist_ok=True)

print("Saving baseline model...")

student_model.save_pretrained(BASELINE_PATH)
student_tokenizer.save_pretrained(BASELINE_PATH)

print("\nBaseline model saved successfully!")
print("Path:")
print(BASELINE_PATH)

print("\nSaved files:")
print(os.listdir(BASELINE_PATH))

Saving baseline model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Baseline model saved successfully!
Path:
/content/drive/MyDrive/SLM_KD_Research/models/baseline_student/final

Saved files:
['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
# ==========================================
# RECONNECT AFTER RUNTIME RESTART
# ==========================================

import os
import torch

print("====================================")
print("SYSTEM CHECK")
print("====================================")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is NOT available.")

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/SLM_KD_Research"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Selected device:", device)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nBaseline checkpoint exists:",
      os.path.exists(
          os.path.join(
              PROJECT_ROOT,
              "models",
              "baseline_student",
              "final",
              "model.safetensors"
          )
      ))

SYSTEM CHECK
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Project root:
/content/drive/MyDrive/SLM_KD_Research

Baseline checkpoint exists: True


In [ ]:
# ==========================================
# INSPECT SST-2 TEST LABELS
# ==========================================

from datasets import load_dataset

print("Loading SST-2...")

dataset = load_dataset(
    "nyu-mll/glue",
    "sst2"
)

print("\nDataset loaded successfully.")

print("\nTest split:")
print(dataset["test"])

print("\nTest label information:")
print(dataset["test"].features["label"])

print("\nFirst 20 test labels:")
print(dataset["test"]["label"][:20])

print("\nUnique test labels:")
print(set(dataset["test"]["label"]))

Loading SST-2...



Dataset loaded successfully.

Test split:
Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 1821
})

Test label information:
ClassLabel(names=['negative', 'positive'])

First 20 test labels:
[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]

Unique test labels:
{-1}


In [ ]:
# ==========================================
# RELOAD SAVED BASELINE STUDENT
# ==========================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

BASELINE_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "baseline_student",
    "final"
)

print("Loading saved Baseline Student...")

baseline_tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_PATH
)

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    BASELINE_PATH
)

baseline_model = baseline_model.to(
    torch.device("cuda")
)

print("\nBaseline Student loaded successfully!")

print("Parameters:",
      sum(p.numel() for p in baseline_model.parameters()))

print("Device:",
      next(baseline_model.parameters()).device)

Loading saved Baseline Student...


Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]


Baseline Student loaded successfully!
Parameters: 4386178
Device: cuda:0


In [ ]:
# ==========================================
# CREATE VALIDATION DATALOADER
# ==========================================

MAX_LENGTH = 128
BATCH_SIZE = 16

def tokenize_validation(example):
    return baseline_tokenizer(
        example["sentence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

print("Tokenizing validation set...")

validation_dataset = dataset["validation"].map(
    tokenize_validation,
    batched=True
)

validation_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "token_type_ids",
        "attention_mask",
        "label"
    ]
)

data_collator = DataCollatorWithPadding(
    tokenizer=baseline_tokenizer
)

baseline_validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print("\nValidation DataLoader created successfully!")

print("Validation examples:",
      len(validation_dataset))

print("Validation batches:",
      len(baseline_validation_loader))

Tokenizing validation set...


Map:   0%|          | 0/872 [00:00<?, ? examples/s]


Validation DataLoader created successfully!
Validation examples: 872
Validation batches: 55


In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [ ]:
# ==========================================
# VERIFY SAVED BASELINE ON VALIDATION SET
# ==========================================

baseline_model.eval()

all_predictions = []
all_labels = []
total_val_loss = 0.0

print("Evaluating saved Baseline Student...")

with torch.no_grad():

    for batch in baseline_validation_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = baseline_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )

        total_val_loss += outputs.loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=-1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

validation_loss = total_val_loss / len(
    baseline_validation_loader
)

validation_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

validation_f1 = f1_score(
    all_labels,
    all_predictions
)

print("\n====================================")
print("SAVED BASELINE VALIDATION RESULTS")
print("====================================")

print(f"Validation Loss:     {validation_loss:.4f}")
print(f"Validation Accuracy: {validation_accuracy:.4f}")
print(f"Validation F1:       {validation_f1:.4f}")

Evaluating saved Baseline Student...

SAVED BASELINE VALIDATION RESULTS
Validation Loss:     0.4272
Validation Accuracy: 0.8291
Validation F1:       0.8269


In [ ]:
# ==========================================
# DIAGNOSE BASELINE PREDICTIONS
# ==========================================

from collections import Counter

print("====================================")
print("PREDICTION / LABEL CHECK")
print("====================================")

print("Unique true labels:")
print(Counter(all_labels))

print("\nUnique predictions:")
print(Counter(all_predictions))

print("\nTotal labels:", len(all_labels))
print("Total predictions:", len(all_predictions))

print("\nAccuracy:", accuracy_score(
    all_labels,
    all_predictions
))

print("\nF1 with binary averaging:")
print(f1_score(
    all_labels,
    all_predictions,
    average="binary"
))

print("\nF1 with macro averaging:")
print(f1_score(
    all_labels,
    all_predictions,
    average="macro"
))

print("\nF1 with weighted averaging:")
print(f1_score(
    all_labels,
    all_predictions,
    average="weighted"
))

PREDICTION / LABEL CHECK
Unique true labels:
Counter({np.int64(1): 444, np.int64(0): 428})

Unique predictions:
Counter({np.int64(0): 455, np.int64(1): 417})

Total labels: 872
Total predictions: 872

Accuracy: 0.8291284403669725

F1 with binary averaging:
0.826945412311266

F1 with macro averaging:
0.8291012452269806

F1 with weighted averaging:
0.8290616886597199


In [ ]:
# ==========================================
# RESTORE BASELINE EXPERIMENT SETTINGS
# ==========================================

STUDENT_MODEL = "google/bert_uncased_L-2_H-128_A-2"
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
SEED = 42

print("Baseline settings restored.")
print("Student:", STUDENT_MODEL)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", NUM_EPOCHS)
print("Seed:", SEED)

Baseline settings restored.
Student: google/bert_uncased_L-2_H-128_A-2
Max length: 128
Batch size: 16
Learning rate: 2e-05
Epochs: 3
Seed: 42


In [ ]:
# ==========================================
# SAVE VERIFIED BASELINE RESULTS
# ==========================================

import os
import json

baseline_results = {
    "experiment": "E1_baseline_student",
    "student_model": STUDENT_MODEL,
    "dataset": "SST-2",
    "split": "validation",
    "validation_samples": len(all_labels),

    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "seed": SEED,

    "total_parameters": 4386178,

    "validation_loss": float(validation_loss),

    "validation_accuracy": float(
        accuracy_score(
            all_labels,
            all_predictions
        )
    ),

    "validation_f1": float(
        f1_score(
            all_labels,
            all_predictions,
            average="binary"
        )
    )
}

RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "baseline_results.json"
)

with open(RESULTS_PATH, "w") as f:
    json.dump(
        baseline_results,
        f,
        indent=4
    )

print("====================================")
print("BASELINE RESULTS SAVED")
print("====================================")

print("File:")
print(RESULTS_PATH)

print("\nResults:")
print(json.dumps(
    baseline_results,
    indent=4
))

BASELINE RESULTS SAVED
File:
/content/drive/MyDrive/SLM_KD_Research/results/baseline_results.json

Results:
{
    "experiment": "E1_baseline_student",
    "student_model": "google/bert_uncased_L-2_H-128_A-2",
    "dataset": "SST-2",
    "split": "validation",
    "validation_samples": 872,
    "max_length": 128,
    "batch_size": 16,
    "learning_rate": 2e-05,
    "epochs": 3,
    "seed": 42,
    "total_parameters": 4386178,
    "validation_loss": 0.4271568989211863,
    "validation_accuracy": 0.8291284403669725,
    "validation_f1": 0.826945412311266
}
